# Lab4: честное разделение algorithm A/B и prompt A/B
Dataset: `docs/quality/golden-dataset/dataset.jsonl`. Локальный прогон ниже не вызывает провайдера. Настоящий prompt-only A/B запускается отдельно с `--mode provider` и ключом.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from ml.experiments.run_ab import load_cases, run_algorithm_ab, run_contract
dataset = ROOT / 'docs/quality/golden-dataset/dataset.jsonl'
cases = load_cases(dataset)
algorithm = run_algorithm_ab(cases)
contract = run_contract(cases)
{'dataset': str(dataset.relative_to(ROOT)), 'algorithm_cases': algorithm['case_count'], 'contract_cases': contract['case_count']}

{'dataset': 'docs/quality/golden-dataset/dataset.jsonl',
 'algorithm_cases': 40,
 'contract_cases': 91}

In [2]:
summary = {
    'A_algorithm_agreement': algorithm['variant_a']['agreement'],
    'B_algorithm_agreement': algorithm['variant_b']['agreement'],
    'delta_pp': algorithm['delta_agreement_pp'],
    'contract_passed': contract['passed'],
    'provider_called': contract['provider_called'],
    'quality_measured_by_contract': contract['quality_measured'],
}
summary

{'A_algorithm_agreement': 0.825,
 'B_algorithm_agreement': 0.9,
 'delta_pp': 7.5,
 'contract_passed': True,
 'provider_called': False,
 'quality_measured_by_contract': False}

In [3]:
errors = [row for row in algorithm['cases'] if not row['b_correct']]
errors[:10]

[{'id': 'REV-001', 'expected': 'correct', 'a': 'partially_correct', 'b': 'partially_correct', 'a_correct': False, 'b_correct': False},
 {'id': 'REV-021', 'expected': 'partially_correct', 'a': 'correct', 'b': 'correct', 'a_correct': False, 'b_correct': False},
 {'id': 'REV-024', 'expected': 'partially_correct', 'a': 'correct', 'b': 'correct', 'a_correct': False, 'b_correct': False},
 {'id': 'REV-039', 'expected': 'partially_correct', 'a': 'correct', 'b': 'correct', 'a_correct': False, 'b_correct': False}]

In [4]:
usage = {
    'v1_estimated_input_tokens': sum(r['estimated_input_tokens']['v1'] for r in contract['cases']),
    'v2_estimated_input_tokens': sum(r['estimated_input_tokens']['v2'] for r in contract['cases']),
    'provider_usage_available': False,
}
usage

{'v1_estimated_input_tokens': 5121,
 'v2_estimated_input_tokens': 15180,
 'provider_usage_available': False}

## Вывод
B выбран только как лучший локальный fallback-алгоритм (90% против 82,5%). Contract check не измеряет качество промпта. Решение о prompt v2 по LLM откладывается до разрешённого provider-прогона с одинаковыми параметрами и несколькими повторами.